## Imports

In [1]:
import googlemaps
import pandas as pd
import numpy as np
import re
import os
import geopandas as gpd
import folium
from datetime import datetime, timedelta
import glob

## Read-in

In [2]:
# Get today's date
today = datetime.now()

# Get the date one week ago
one_week_ago = today - timedelta(days=7)

# Function to format the date with the correct ordinal suffix
def format_date_with_ordinal(date):
    day = date.day
    if 4 <= day <= 20 or 24 <= day <= 30:
        suffix = "th"
    else:
        suffix = ["st", "nd", "rd"][day % 10 - 1]
    return date.strftime(f"%b. {day}{suffix}")

# Format both dates
week_ending = format_date_with_ordinal(today)
week_starting = format_date_with_ordinal(one_week_ago)

print(week_starting, week_ending)

Jul. 15th Jul. 22nd


In [3]:
filename = glob.glob('redfin*.csv')
df_list = []
for file in filename:
    df = pd.read_csv(file)
    df_list.append(df)
    
df = pd.concat(df_list)

In [4]:
df = df.rename(columns={'URL (SEE https://www.redfin.com/buy-a-home/comparative-market-analysis FOR INFO ON PRICING)':'URL'})

In [5]:
remove_string = 'In accordance with local MLS rules, some MLS listings are not included in the download'
df = df[~df['SALE TYPE'].str.contains(remove_string)]

In [6]:
df['SOLD DATE'] = pd.to_datetime(df['SOLD DATE'])

In [8]:
# # Filter dates between. If only using data looking back one week, this shouldn't change the df
# df = df[(df['SOLD DATE'] >= '2024-06-10') & (df['SOLD DATE'] <= '2024-06-17')]

In [7]:
df.sort_values(ascending=False, by='SOLD DATE')

,SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,...,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL,SOURCE,MLS#,FAVORITE,INTERESTED,LATITUDE,LONGITUDE
1,PAST SALE,2024-07-19,Condo/Co-op,3601 SW 11th St Unit 2B,Miami,FL,33135.0,245000.0,2.0,1.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/3601-SW-11th-S...,MARMLS,A11604950,N,Y,25.762756,-80.252335
87,PAST SALE,2024-07-19,Condo/Co-op,1351 NE Miami Gardens Dr Unit 1101E,Miami,FL,33179.0,145000.0,1.0,1.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/1351-NE-Miami-...,MARMLS,A11527085,N,Y,25.945838,-80.174826
44,PAST SALE,2024-07-19,Condo/Co-op,9359 Fontainebleau Blvd Unit F417,Miami,FL,33172.0,320000.0,2.0,2.5,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/9359-Fontaineb...,MARMLS,A11601997,N,Y,25.774834,-80.345221
51,PAST SALE,2024-07-19,Condo/Co-op,730 Pennsylvania Ave #405,Miami Beach,FL,33139.0,275000.0,1.0,1.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/730-Penn...,MARMLS,A11596214,N,Y,25.777794,-80.134219
62,PAST SALE,2024-07-19,Condo/Co-op,19195 Mystic Pointe Dr #1507,Aventura,FL,33180.0,357000.0,1.0,1.5,...,Sold,NaN,NaN,https://www.redfin.com/FL/Aventura/19195-Mysti...,MARMLS,A11579744,N,Y,25.953453,-80.126851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,PAST SALE,2024-07-15,Condo/Co-op,250 NW 107th Ave #204,Miami,FL,33172.0,290000.0,2.0,2.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/250-NW-107th-A...,MARMLS,A11584033,N,Y,25.771778,-80.369582
34,PAST SALE,2024-07-15,Condo/Co-op,8200 SW 210th St #109,Cutler Bay,FL,33189.0,270000.0,2.0,2.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Cutler-Bay/8200-SW-2...,MARMLS,A11607767,N,Y,25.571760,-80.323328
69,PAST SALE,2024-07-15,Condo/Co-op,1491 SW 124th Ct Unit 8-D,Miami,FL,33184.0,450000.0,3.0,2.5,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/1491-SW-124th-...,MARMLS,A11564507,N,Y,25.754036,-80.395880
22,PAST SALE,2024-07-15,Condo/Co-op,1780 NE 191st St Unit 314-2,Miami,FL,33179.0,155000.0,2.0,2.0,...,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/1780-NE-191st-...,MARMLS,A11579601,N,Y,25.949923,-80.166133


In [9]:
df = df.sort_values(by='PRICE',ascending=False)
df_top_ten = df.head(10)

## Current Week's Values

In [10]:
# Number of condos sold
current_week_condo_sales = len(df)
print(len(df))

140


In [11]:
# Average sale price
average_sale_price = round(df['PRICE'].mean())
print(f'${average_sale_price:,.0f}')

$938,812


In [12]:
# Average PSF
average_price_psf = round(df['$/SQUARE FEET'].mean())
print(f'${average_price_psf:,.0f}')

$586


In [13]:
# Total sales dollar volume
current_week_condo_sales_total = df['PRICE'].sum()
print(f'${current_week_condo_sales_total:,.0f}')

$131,433,720


In [14]:
# # Define the date range
# start_date = '2024-05-20'
# end_date = '2024-05-26'

# # Filter rows where 'SOLD DATE' is within the specified range
# second_week_df = df[(df['SOLD DATE'] >= start_date) & (df['SOLD DATE'] <= end_date)]

## Previous Week's Values

In [15]:
print('Input Previous Week Condo Sales Total (number units sold):')
previous_week_condo_sales = input()
print('Input Previous Week Condo Average Sales Price:')
previous_week_average_sales_price = input()
print('Input Previous Week Condo Average PSF:')
previous_week_average_psf = input()
print('Input Previous Week Condo Sales total (ex: 198_000_000)')
previous_week_condo_sales_total = input()

Input Previous Week Condo Sales Total (number units sold):
113
Input Previous Week Condo Average Sales Price:
821936
Input Previous Week Condo Average PSF:
563
Input Previous Week Condo Sales total (ex: 198_000_000)
92_800_000


## Clean Data

In [36]:
# df_top_ten

## Correction section

In [10]:
# df.at[index#,'col_name']

## Format Data

In [16]:
### Insert NaNs if needed ###
df_top_ten = df_top_ten.replace('N/A', np.nan)

## Color-code top sale

In [17]:
### Insert RANK values ###
df_top_ten['RANK'] = range(1, len(df_top_ten) + 1)
# use numpy to assign values to the 'COLOR' column
df_top_ten['COLOR'] = np.where(df_top_ten['RANK'] <= 1, 'orange', 'blue')

## HTML Popup Formatter

In [18]:
df_top_ten.columns

Index(['SALE TYPE', 'SOLD DATE', 'PROPERTY TYPE', 'ADDRESS', 'CITY',
       'STATE OR PROVINCE', 'ZIP OR POSTAL CODE', 'PRICE', 'BEDS', 'BATHS',
       'LOCATION', 'SQUARE FEET', 'LOT SIZE', 'YEAR BUILT', 'DAYS ON MARKET',
       '$/SQUARE FEET', 'HOA/MONTH', 'STATUS', 'NEXT OPEN HOUSE START TIME',
       'NEXT OPEN HOUSE END TIME', 'URL', 'SOURCE', 'MLS#', 'FAVORITE',
       'INTERESTED', 'LATITUDE', 'LONGITUDE', 'RANK', 'COLOR'],
      dtype='object')

In [19]:
pd.set_option('display.max_columns',None)

In [20]:
df_top_ten.head(1)

,SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,LOCATION,SQUARE FEET,LOT SIZE,YEAR BUILT,DAYS ON MARKET,$/SQUARE FEET,HOA/MONTH,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL,SOURCE,MLS#,FAVORITE,INTERESTED,LATITUDE,LONGITUDE,RANK,COLOR
55,PAST SALE,2024-07-18,Condo/Co-op,100 S Pointe Dr #3507,Miami Beach,FL,33139.0,14770000.0,4.0,4.0,CONTINUUM ON SOUTH BEACH,2954.0,NaN,2003.0,NaN,5000.0,5683.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/100-S-Po...,MARMLS,A11571861,N,Y,25.767161,-80.132776,1,orange


In [21]:
def popup_html(row):
    building_name = row['LOCATION']
    price = row['PRICE']
    psf = row['$/SQUARE FEET']
    sf = row['SQUARE FEET']
    year_built = row['YEAR BUILT']
    
    html = '''<!DOCTYPE html>
    <html>
    <strong>Building Name: </strong>{}'''.format(building_name) + '''<br>
    <strong>Sale Price: </strong>${}'''.format(f'{price:,}') + '''<br>
    <strong>Price sq ft: </strong>${}'''.format(f'{psf:,}') + '''<br>
    <strong>Square Footage: </strong>{}'''.format(f'{sf:,}') + '''<br>
    <strong>Year Built: </strong>{}'''.format(f'{str(year_built)}') + '''<br>
    </html>
    '''
    return html

In [22]:
from folium.plugins import MarkerCluster

title_html = '''
              <h3 align="center" style="font-size:16px"><b>{}</b></h3>
             '''.format(f'Recent Miami-Dade Condo Sales ')

caption_html = '''
                <p align="center" style="vertical-align: bottom; font-size:13px"><i>{}</i></p>
                '''.format(f'{week_starting} - {week_ending}')


### Create map container ###
m = folium.Map(location=df[["LATITUDE", "LONGITUDE"]].mean().to_list(),zoom_start=9.5,tiles=None)

# Create two FeatureGroups for different color pins
fg_blue = folium.FeatureGroup(name='All other sales')
fg_orange = folium.FeatureGroup(name='Top Sale')

for index, row in df_top_ten.iterrows():
    # Add the markers to the appropriate FeatureGroup based on the color
    if row['COLOR'] == 'blue':
        marker = folium.Marker(
            location=[row['LATITUDE'], row['LONGITUDE']],
            radius=5,
            fill=True,
            icon=folium.Icon(color=row['COLOR']),
            popup=folium.Popup(popup_html(row), max_width=400))
        marker.add_to(fg_blue)
    else:
        marker = folium.Marker(
            location=[row['LATITUDE'], row['LONGITUDE']],
            radius=5,
            fill=True,
            icon=folium.Icon(color=row['COLOR']),
            popup=folium.Popup(popup_html(row), max_width=400))
        marker.add_to(fg_orange)

# Add the FeatureGroups to the map
fg_orange.add_to(m)
fg_blue.add_to(m)

folium.TileLayer('OpenStreetMap',control=False).add_to(m)

# Add LayerControl to the map
folium.map.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(caption_html))
            
# Display map
m

In [23]:
m.save('index.html')

## Data snagger

In [24]:
### Set up formatting ###
BR = '\n'

ME = '\033[1m' + 'Most Expensive' + '\033[0m'
LE = '\033[1m' + 'Least Expensive' + '\033[0m'

MAX_PSF = '\033[1m' + 'Highest Price Per Square Foot' + '\033[0m'
MIN_PSF = '\033[1m' + 'Lowest Price Per Square Foot' + '\033[0m'

DAYS_MAX = '\033[1m' + 'Most Days on Market' + '\033[0m'
DAYS_MIN = '\033[1m' + 'Fewest Days on Market' + '\033[0m'

In [25]:
df_top_ten.head(1)

,SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,LOCATION,SQUARE FEET,LOT SIZE,YEAR BUILT,DAYS ON MARKET,$/SQUARE FEET,HOA/MONTH,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL,SOURCE,MLS#,FAVORITE,INTERESTED,LATITUDE,LONGITUDE,RANK,COLOR
55,PAST SALE,2024-07-18,Condo/Co-op,100 S Pointe Dr #3507,Miami Beach,FL,33139.0,14770000.0,4.0,4.0,CONTINUUM ON SOUTH BEACH,2954.0,NaN,2003.0,NaN,5000.0,5683.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/100-S-Po...,MARMLS,A11571861,N,Y,25.767161,-80.132776,1,orange


In [ ]:
# , {df_top_ten[df_top_ten['PRICE'].idxmax()]['ADDRESS']} | Price ${df_top_ten[df_top_ten['PRICE'].idxmax()]['PRICE']:,.0f}

In [26]:
df_top_ten['PRICE'].iloc[0]

14770000.0

In [122]:
# # Highest and lowest sale price
# highest_sale_price_idx = df_top_ten['PRICE'].idxmax()
# lowest_sale_price_idx = df_top_ten['PRICE'].idxmin()

# print(f"Highest Sale Price: {df_top_ten.loc[highest_sale_price_idx]['LOCATION']}, {df_top_ten.loc[highest_sale_price_idx]['ADDRESS']} | Price: ${df_top_ten.loc[highest_sale_price_idx]['PRICE']:,.0f} | ${df_top_ten.loc[highest_sale_price_idx]['$/SQUARE FEET']:,.0f} psf | Days on market: {df_top_ten.loc[highest_sale_price_idx]['DAYS ON MARKET']}")
# print(f"Lowest Sale Price: {df_top_ten.loc[lowest_sale_price_idx]['LOCATION']}, {df_top_ten.loc[lowest_sale_price_idx]['ADDRESS']} | Price: ${df_top_ten.loc[lowest_sale_price_idx]['PRICE']:,.0f} | ${df_top_ten.loc[lowest_sale_price_idx]['$/SQUARE FEET']:,.0f} psf | Days on market: {df_top_ten.loc[lowest_sale_price_idx]['DAYS ON MARKET']}")

# # Highest and lowest psf
# highest_psf_idx = df_top_ten['$/SQUARE FEET'].idxmax()
# lowest_psf_idx = df_top_ten['$/SQUARE FEET'].idxmin()

# print(f"Highest PSF: {df_top_ten.loc[highest_psf_idx]['LOCATION']}, {df_top_ten.loc[highest_psf_idx]['ADDRESS']} | Price: ${df_top_ten.loc[highest_psf_idx]['PRICE']:,.0f} | ${df_top_ten.loc[highest_psf_idx]['$/SQUARE FEET']:,.0f} psf | Days on market: {df_top_ten.loc[highest_psf_idx]['DAYS ON MARKET']}")
# print(f"Lowest PSF: {df_top_ten.loc[lowest_psf_idx]['LOCATION']}, {df_top_ten.loc[lowest_psf_idx]['ADDRESS']} | Price: ${df_top_ten.loc[lowest_psf_idx]['PRICE']:,.0f} | ${df_top_ten.loc[lowest_psf_idx]['$/SQUARE FEET']:,.0f} psf | Days on market: {df_top_ten.loc[lowest_psf_idx]['DAYS ON MARKET']}")

## Map URL snagger

Map template URL: `https://trd-digital.github.io/trd-news-interactive-maps/{map-folder-name}`

In [27]:
base_name = 'https://trd-digital.github.io/trd-news-interactive-maps/'

cwd = os.getcwd()

cwd = cwd.split('/')

final_name = base_name + cwd[-1]
print(final_name)

https://trd-digital.github.io/trd-news-interactive-maps/condo_sales_week_ending_07222024


# CREATE TEMPLATE 

In [28]:
muni_set = set(df_top_ten['CITY'])

In [29]:
df_top_ten['FULL_NAME'] = df_top_ten['LOCATION'] + ' at ' + df_top_ten['ADDRESS'] + ' in ' + df_top_ten['CITY']

In [30]:
df_top_ten.reset_index(inplace=True,drop=True)

In [31]:
top_sale = df_top_ten.at[0,'FULL_NAME']
second_top_sale = df_top_ten.at[1,'FULL_NAME']

In [32]:
top_sale

'CONTINUUM ON SOUTH BEACH at 100 S Pointe Dr #3507 in Miami Beach'

In [33]:
### Average Sales Price ###
if float(average_sale_price) > float(previous_week_average_sales_price):
    change = 'higher'
    difference = float(average_sale_price) - float(previous_week_average_sales_price)
else:
    change = 'lower'
    difference = float(previous_week_average_sales_price) - float(average_sale_price)
    
### Total condo dollar volume ###
if float(current_week_condo_sales_total) > float(previous_week_condo_sales_total):
    seo_head = 'Rises'
elif float(previous_week_condo_sales_total) < float(current_week_condo_sales_total):
    seo_head = 'Falls'
else:
    seo_head = 'Unchanged'
    
### Average PSF ###    
if float(average_price_psf) > float(previous_week_average_psf):
    psf_rf = 'rose'
elif float(average_price_psf) < float(previous_week_average_psf):
    psf_rf ='fell'
else:
    psf_rf = 'equaled'

In [34]:
# df_top_ten

## RESUME HERE

In [34]:
story_string = f'''
\033[1mHED:\033[0m {df_top_ten.loc[df_top_ten['PRICE'].idxmax()]['LOCATION']} closing tops Miami-Dade weekly condo sales 
\033[1mDEK:\033[0m Top 10 sales ranged from ${df_top_ten.at[9,'PRICE']:,.0f}M to ${df_top_ten.at[0,'PRICE']:,.0f}M
\033[1mFEATURED HED:\033[0m
\033[1mSEO HED:\033[0m Miami-Dade County Weekly Condo Report 
\033[1mSEO DESCRIPTION:\033[0m Miami-Dade County’s condo sales XXXXXXX
\033[1mAUTHOR:\033[0m Adam Farence
\033[1mRESEARCH:\033[0m 
\033[1mSocial:\033[0m #MiamiDade condo sales XXXXX
\033[1mART:\033[0m

Interactive map

*Please provide credits for any images that you share
\033[1mSTORY TYPE:\033[0m Report
\033[1mSECTOR\033[0m (formerly CATEGORY): Residential Real Estate
\033[1mTAGS:\033[0m condo sales, Miami-Dade County, weekly condo sales, {', '.join(str(x) for x in muni_set)}

\033[1mNeighborhood:\033[0m 
\033[1mProperty:\033[0m
\033[1mProperty Type:\033[0m
\033[1mCompanies:\033[0m 
\033[1mPeople:\033[0m
\033[1mIssues:\033[0m
\033[1mRegion:\033[0m
\033[1m[]Show in Yahoo Feed\033[0m

Miami-Dade County’s NEWS PEG HERE.

Brokers closed {int(current_week_condo_sales):,.0f} condo sales totaling ${float(current_week_condo_sales_total):,.0f} million from {week_starting} to {week_ending}. The previous week, brokers closed {previous_week_condo_sales} condo sales totaling ${int(previous_week_condo_sales_total):,.0f}.

Last week’s units sold for an average of ${float(average_sale_price):,.0f}, {change} than the ${float(previous_week_average_sales_price):,.0f} average price from the previous week. The average price per square foot {psf_rf} to ${average_price_psf} from ${previous_week_average_psf}, according to data from Redfin.

For the top 10 sales, prices ranged from ${df_top_ten.at[9,'PRICE']:,} million to ${df_top_ten.at[0,'PRICE']:,} million. 

{top_sale} took the top spot with a ${df_top_ten.at[0,'PRICE']:,} million closing. The sale closed at ${df_top_ten.at[0,'$/SQUARE FEET']:,} per square foot.

{second_top_sale}, closed for the second highest amount, ${df_top_ten.at[1,'PRICE']:,}, or ${df_top_ten.at[1,'$/SQUARE FEET']:,} per square foot.

<figure>
 <div class="container">
   <div class="iframe-wrap">
   <iframe src="{final_name}" width="100%" height="600" frameBorder="0" scrolling="no"></iframe>
  </div>
</div>
  <figcaption align="right"><a href="https://leafletjs.com/">Leaflet</a> map created by Adam Farence | Data by © <a href="https://www.openstreetmap.org/#map=4/38.01/-95.84"> OpenStreetMap</a>, under <a href="https://www.openstreetmap.org/copyright">ODbl.</a></figcaption>
</figure>

Here’s a breakdown of the top 10 sales from {week_starting} to {week_ending}:
'''

print(story_string)


HED: CONTINUUM ON SOUTH BEACH closing tops Miami-Dade weekly condo sales 
DEK: Top 10 sales ranged from $2,600,000M to $14,770,000M
FEATURED HED:
SEO HED: Miami-Dade County Weekly Condo Report 
SEO DESCRIPTION: Miami-Dade County’s condo sales XXXXXXX
AUTHOR: Adam Farence
RESEARCH: 
Social: #MiamiDade condo sales XXXXX
ART:

Interactive map

*Please provide credits for any images that you share
STORY TYPE: Report
SECTOR (formerly CATEGORY): Residential Real Estate
TAGS: condo sales, Miami-Dade County, weekly condo sales, Sunny Isles Beach, Key Biscayne, Bal Harbour, Miami Beach, Miami

Neighborhood: 
Property:
Property Type:
Companies: 
People:
Issues:
Region:
[]Show in Yahoo Feed

Miami-Dade County’s NEWS PEG HERE.

Brokers closed 140 condo sales totaling $131,433,720 million from Jul. 15th to Jul. 22nd. The previous week, brokers closed 113 condo sales totaling $92,800,000.

Last week’s units sold for an average of $938,812, higher than the $821,936 sales average from the previous we

In [35]:
story_checklist = '''
\033[1mRemember to...:\033[0m

1. Double check all names. Sometimes names differ between the hed and the body of the story.
    For example, "Surf Club Four Seasons" in the HED and "Four Seasons Residences at the Surfclub"
    in the body.
    
2. Add in context, if available. When there's a high-priced condo sale, check and see if there's
    a story. If there is, add in some extra details and link back to it.
'''

In [36]:
### Set up formatting ###
BR = '\n'

ME = '\033[1m' + 'Most Expensive' + '\033[0m'
LE = '\033[1m' + 'Least Expensive' + '\033[0m'

MAX_PSF = '\033[1m' + 'Highest Price Per Square Foot' + '\033[0m'
MIN_PSF = '\033[1m' + 'Lowest Price Per Square Foot' + '\033[0m'

In [37]:
print(story_string)

# Highest and lowest sale price
highest_sale_price_idx = df_top_ten['PRICE'].idxmax()
lowest_sale_price_idx = df_top_ten['PRICE'].idxmin()

print(f"{ME}{BR}{df_top_ten.loc[highest_sale_price_idx]['LOCATION']}, {df_top_ten.loc[highest_sale_price_idx]['ADDRESS']} in {df_top_ten.loc[highest_sale_price_idx]['CITY']} | Price: ${df_top_ten.loc[highest_sale_price_idx]['PRICE']:,.0f} | ${df_top_ten.loc[highest_sale_price_idx]['$/SQUARE FEET']:,.0f} psf")
print(f"{LE}{BR}{df_top_ten.loc[lowest_sale_price_idx]['LOCATION']}, {df_top_ten.loc[lowest_sale_price_idx]['ADDRESS']} in {df_top_ten.loc[lowest_sale_price_idx]['CITY']} | Price: ${df_top_ten.loc[lowest_sale_price_idx]['PRICE']:,.0f} | ${df_top_ten.loc[lowest_sale_price_idx]['$/SQUARE FEET']:,.0f} psf")

# Highest and lowest psf
highest_psf_idx = df_top_ten['$/SQUARE FEET'].idxmax()
lowest_psf_idx = df_top_ten['$/SQUARE FEET'].idxmin()

print(f"{MAX_PSF}{BR}{df_top_ten.loc[highest_psf_idx]['LOCATION']}, {df_top_ten.loc[highest_psf_idx]['ADDRESS']} in {df_top_ten.loc[highest_psf_idx]['CITY']} | Price: ${df_top_ten.loc[highest_psf_idx]['PRICE']:,.0f} | ${df_top_ten.loc[highest_psf_idx]['$/SQUARE FEET']:,.0f} psf")
print(f"{MIN_PSF}{BR}{df_top_ten.loc[lowest_psf_idx]['LOCATION']}, {df_top_ten.loc[lowest_psf_idx]['ADDRESS']} in {df_top_ten.loc[lowest_psf_idx]['CITY']} | Price: ${df_top_ten.loc[lowest_psf_idx]['PRICE']:,.0f} | ${df_top_ten.loc[lowest_psf_idx]['$/SQUARE FEET']:,.0f} psf")

print(story_checklist)


HED: CONTINUUM ON SOUTH BEACH closing tops Miami-Dade weekly condo sales 
DEK: Top 10 sales ranged from $2,600,000M to $14,770,000M
FEATURED HED:
SEO HED: Miami-Dade County Weekly Condo Report 
SEO DESCRIPTION: Miami-Dade County’s condo sales XXXXXXX
AUTHOR: Adam Farence
RESEARCH: 
Social: #MiamiDade condo sales XXXXX
ART:

Interactive map

*Please provide credits for any images that you share
STORY TYPE: Report
SECTOR (formerly CATEGORY): Residential Real Estate
TAGS: condo sales, Miami-Dade County, weekly condo sales, Sunny Isles Beach, Key Biscayne, Bal Harbour, Miami Beach, Miami

Neighborhood: 
Property:
Property Type:
Companies: 
People:
Issues:
Region:
[]Show in Yahoo Feed

Miami-Dade County’s NEWS PEG HERE.

Brokers closed 140 condo sales totaling $131,433,720 million from Jul. 15th to Jul. 22nd. The previous week, brokers closed 113 condo sales totaling $92,800,000.

Last week’s units sold for an average of $938,812, higher than the $821,936 sales average from the previous we

In [38]:
print(df_top_ten['URL'].iloc[0])

https://www.redfin.com/FL/Miami-Beach/100-S-Pointe-Dr-33139/unit-3507/home/43271133


In [39]:
print(df_top_ten['URL'].iloc[1])

https://www.redfin.com/FL/Sunny-Isles-Beach/17975-Collins-Ave-33160/unit-4402/home/190192461


In [40]:
print(df_top_ten['URL'].iloc[-1])

https://www.redfin.com/FL/Key-Biscayne/615-Ocean-Dr-33149/unit-8B/home/42914670


In [41]:
print(df_top_ten['URL'].iloc[-2])

https://www.redfin.com/FL/Bal-Harbour/9801-Collins-Ave-33154/unit-17Z/home/42905426


In [47]:
# Find the index of the row with the maximum '$/SQUARE FEET' value
max_index = df_top_ten['$/SQUARE FEET'].idxmax()

# Use the index to access the 'URL' of that row
max_url = df_top_ten.loc[max_index, 'URL']

print(max_url)

https://www.redfin.com/FL/Miami-Beach/100-S-Pointe-Dr-33139/unit-3507/home/43271133


In [48]:
# Find the index of the row with the minimum '$/SQUARE FEET' value
min_index = df_top_ten['$/SQUARE FEET'].idxmin()

# Use the index to access the 'URL' of that row
min_url = df_top_ten.loc[min_index, 'URL']

print(min_url)

https://www.redfin.com/FL/Sunny-Isles-Beach/330-Sunny-Isles-Blvd-33160/unit-2303/home/184722534


In [44]:
df_top_ten

,SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,LOCATION,SQUARE FEET,LOT SIZE,YEAR BUILT,DAYS ON MARKET,$/SQUARE FEET,HOA/MONTH,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL,SOURCE,MLS#,FAVORITE,INTERESTED,LATITUDE,LONGITUDE,RANK,COLOR,FULL_NAME
0,PAST SALE,2024-07-18,Condo/Co-op,100 S Pointe Dr #3507,Miami Beach,FL,33139.0,14770000.0,4.0,4.0,CONTINUUM ON SOUTH BEACH,2954.0,NaN,2003.0,NaN,5000.0,5683.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/100-S-Po...,MARMLS,A11571861,N,Y,25.767161,-80.132776,1,orange,CONTINUUM ON SOUTH BEACH at 100 S Pointe Dr #3...
1,PAST SALE,2024-07-16,Condo/Co-op,17975 Collins Ave #4402,Sunny Isles Beach,FL,33160.0,10500000.0,4.0,5.5,Estates at Acqualina,4385.0,NaN,2023.0,NaN,2395.0,8590.0,Sold,NaN,NaN,https://www.redfin.com/FL/Sunny-Isles-Beach/17...,MARMLS,A11550538,N,Y,25.942332,-80.120166,2,blue,Estates at Acqualina at 17975 Collins Ave #440...
2,PAST SALE,2024-07-15,Condo/Co-op,1000 S Pointe Dr #3402,Miami Beach,FL,33139.0,5500000.0,3.0,3.5,MURANO AT PORTOFINO CONDO,2618.0,NaN,2002.0,NaN,2101.0,5840.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/1000-S-P...,MARMLS,A11258750,N,Y,25.768527,-80.137591,3,blue,MURANO AT PORTOFINO CONDO at 1000 S Pointe Dr ...
3,PAST SALE,2024-07-15,Condo/Co-op,1000 S Pointe Dr #2803,Miami Beach,FL,33139.0,4400000.0,3.0,2.5,Murano At Portofino Condo,2008.0,NaN,2002.0,NaN,2191.0,4789.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/1000-S-P...,MARMLS,A11371347,N,Y,25.768527,-80.137591,4,blue,Murano At Portofino Condo at 1000 S Pointe Dr ...
4,PAST SALE,2024-07-16,Condo/Co-op,5245 Fisher Island Dr #5245,Miami Beach,FL,33109.0,4300000.0,3.0,3.5,BAYVIEW FISHER ISL CONDO,1950.0,NaN,1992.0,NaN,2205.0,6974.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami-Beach/5245-Fis...,MARMLS,A11394274,N,Y,25.762689,-80.148415,5,blue,BAYVIEW FISHER ISL CONDO at 5245 Fisher Island...
5,PAST SALE,2024-07-16,Condo/Co-op,2020 N Bayshore Dr Ph 3906,Miami,FL,33137.0,4000000.0,5.0,5.5,PARAMOUNT BAY CONDO,3996.0,NaN,2010.0,NaN,1001.0,4921.0,Sold,NaN,NaN,https://www.redfin.com/FL/Miami/2020-N-Bayshor...,MARMLS,A11428509,N,Y,25.796683,-80.187669,6,blue,PARAMOUNT BAY CONDO at 2020 N Bayshore Dr Ph 3...
6,PAST SALE,2024-07-15,Condo/Co-op,18555 Collins Ave #3403,Sunny Isles Beach,FL,33160.0,3900000.0,3.0,4.5,18555 COLLINS AVENUE COND,3171.0,NaN,2016.0,NaN,1230.0,6252.0,Sold,NaN,NaN,https://www.redfin.com/FL/Sunny-Isles-Beach/18...,MARMLS,A11466252,N,Y,25.948195,-80.120410,7,blue,18555 COLLINS AVENUE COND at 18555 Collins Ave...
7,PAST SALE,2024-07-17,Condo/Co-op,330 Sunny Isles Blvd #2303,Sunny Isles Beach,FL,33160.0,3050000.0,4.0,4.5,Parque Towers,3094.0,NaN,2019.0,NaN,986.0,2079.0,Sold,NaN,NaN,https://www.redfin.com/FL/Sunny-Isles-Beach/33...,Beaches MLS,F10370724,N,Y,25.929464,-80.125952,8,blue,Parque Towers at 330 Sunny Isles Blvd #2303 in...
8,PAST SALE,2024-07-15,Condo/Co-op,9801 Collins Ave Unit 17Z Direct Ocean,Bal Harbour,FL,33154.0,2800000.0,3.0,3.0,BALMORAL CONDO,1984.0,NaN,1977.0,NaN,1411.0,4880.0,Sold,NaN,NaN,https://www.redfin.com/FL/Bal-Harbour/9801-Col...,MARMLS,A11571005,N,Y,25.890231,-80.122819,9,blue,BALMORAL CONDO at 9801 Collins Ave Unit 17Z Di...
9,PAST SALE,2024-07-19,Condo/Co-op,615 Ocean Dr Unit 8B,Key Biscayne,FL,33149.0,2600000.0,3.0,3.0,SANDS OF KEY BISCAYNE CON,1772.0,NaN,1969.0,NaN,1467.0,2842.0,Sold,NaN,NaN,https://www.redfin.com/FL/Key-Biscayne/615-Oce...,MARMLS,A11527876,N,Y,25.690125,-80.157473,10,blue,SANDS OF KEY BISCAYNE CON at 615 Ocean Dr Unit...


## Time on Market Calculator

In [46]:
################ YEAR, MONTH, DAY #######################

date1 = datetime(2024, 3, 16) ## List (Earlier) date
date2 = datetime(2024, 7, 16) ## Close (Later) date

delta = date2 - date1
num_days = delta.days

print(num_days)

122
